In [1]:
import os
import boto3
from sagemaker import get_execution_role
import time
from pprint import pprint
import shutil

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# name
str_function_name = 'genxii-ad-get-n-feats'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
boto3==1.24.59

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import json
import boto3

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_model = '01_ad'
    
    # load output from concat sensitivity
    print('Loading output from sensitivity analysis concatenation...')
    str_filename = 'df_sensitivity.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/05_lambda_concat_sensitivity/{str_filename}'
    df = pd.read_csv(str_uri)
    
    # subset to features that helped if removed
    print('Subsetting to just those features that helped the model if removed...')
    df = df[df['helped'] == 1].copy()
    
    # get the number of features
    print('Getting the number of features that helped the model once removed...')
    int_n_feats_helped = df.shape[0]
    
    # write to s3 as json for map in step function
    print('Writing json of number of features in model to s3...')
    str_n_feats_helped = json.dumps(int_n_feats_helped)
    cls_client_s3 = boto3.client('s3')
    str_filename = 'json_n_cols_in_model.json'
    str_key = f'{str_model}/02_model/02_model/06_lambda_get_n_feats_for_choice/{str_filename}'
    cls_client_s3.put_object(
        Bucket=str_project,
        Key=str_key,
        Body=str_n_feats_helped,
    )
    
    # return int_n_feats_helped
    return str_n_feats_helped

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-ad-get-n-feats

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  126.5kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 3cd81ffec4d9
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 4bcfff277221
Step 3/6 : COPY requirements.txt  .
 ---> 11afb3582ac5
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Running in 26a8e95271fd
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━

Removing intermediate container 26a8e95271fd
 ---> 96f62d414d27
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> 1fcfe6aa4674
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 2ace49b09b28
Removing intermediate container 2ace49b09b28
 ---> f8207e66bd29
Successfully built f8207e66bd29
Successfully tagged genxii-ad-get-n-feats:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-ad-get-n-feats' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-get-n-feats]
d73252831211: Preparing
36f34554673d: Preparing
f2ca5957a8f5: Preparing
5dc5ed1e2a09: Preparing
e92756f7b561: Preparing
4fe51bf0bf5c: Preparing
fbbd8c1e2ec1: Preparing
fe2359fe88f2: Preparing
e703f2e518cc: Preparing
97a787951169: Preparing
fe2359fe88f2: Waiting
e703f2e518cc: Waiting
4fe51bf0bf5c: Waiting
97a787951169: Waiting
fbbd8c1e2ec1: Waiting
5dc5ed1e2a09: Layer already exists
e92756f7b561: Layer already exists
4fe51bf0bf5c: Layer already exists
fbbd8c1e2ec1: Layer already exists
fe2359fe88f2: Layer already exists
e703f2e518cc: Layer already exists
97a787951169: Layer already exists
f2ca5957a8f5: Pushed
d73252831211: Pushed
36f34554673d: Pushed
latest: digest: sha256:b50e16c959bf3a8f2fc6695c700abf2140d2a6b2071176a07d001f1017f41d6b size: 2419


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 23 Apr 2024 01:48:29 GMT',
                                      'x-amzn-requestid': '528ba59d-50d8-40ad-bf4d-3fa7c3336f79'},
                      'HTTPStatusCode': 204,
                      'RequestId': '528ba59d-50d8-40ad-bf4d-3fa7c3336f79',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-get-n-feats:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': 'b50e16c959bf3a8f2fc6695c700abf2140d2a6b2071176a07d001f1017f41d6b',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-get-n-feats',
 'FunctionName': 'genxii-ad-get-n-feats',
 'LastModified': '2024-04-23T01:48:30.050+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-ad-get-n-feats'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1195',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 23 Apr 2024 01:48:30 GMT',
                                      'x-amzn-requestid': 'dc690c5f-6f92-48c2-bbcc-5a53d70ac781'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'dc690c5f-6f92-48c2-

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)